# Stage 2: cleaning

This notebook turns the raw OpenF1 files in `data/raw/` into clean tables in `data/processed/`.

**Part 1 (this part): one lap table and one pit-stop table.**

Principle: *flag, don't delete*. Every lap is kept and gets flags, because later stages need different subsets (the tyre model needs clean laps, the overtaking model needs every lap). Only rows that are genuinely broken are removed, and they are counted in the cleaning table.

Input: `data/raw/` (from `01_get_data.py`). Output: `data/processed/laps.parquet`, `data/processed/pit_stops.parquet`, `results/cleaning_table.csv`, `results/stint_change_exceptions.csv`.

In [1]:
import time

import numpy as np
import pandas as pd

from helpers import DATA_PROCESSED, RESULTS, load_races, load_raw

start_time = time.time()

# One car in one race.
KEY = ["session_key", "driver_number"]
LAP_KEY = KEY + ["lap_number"]

# A lap end time (next lap's start) that differs from start + lap_duration
# by more than this is flagged. Timing is to the millisecond, so 0.5 s is
# not a rounding difference.
TIME_MISMATCH_SECONDS = 0.5

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

# The cleaning table: one row per step.
cleaning_steps = []


def add_step(step, laps_before, laps_affected, action, reason):
    """Add one row to the cleaning table."""
    cleaning_steps.append(
        {"step": step, "laps_before": int(laps_before), "laps_affected": int(laps_affected),
         "action": action, "reason": reason}
    )

## 1. Load the raw tables

The raw files keep the values as delivered, so column types differ between races. `load_raw` (in `helpers.py`) converts the listed columns explicitly while combining the races.

In [2]:
races = load_races()

laps_raw = load_raw(
    "laps",
    columns=["session_key", "driver_number", "lap_number", "date_start", "lap_duration",
             "duration_sector_1", "duration_sector_2", "duration_sector_3",
             "i1_speed", "i2_speed", "st_speed", "is_pit_out_lap"],
    number_columns=["lap_duration", "duration_sector_1", "duration_sector_2", "duration_sector_3",
                    "i1_speed", "i2_speed", "st_speed"],
    date_columns=["date_start"],
)
drivers = load_raw("drivers", columns=["session_key", "driver_number", "name_acronym", "team_name"])
results = load_raw(
    "session_result",
    columns=["session_key", "driver_number", "position", "number_of_laps", "dnf", "dns", "dsq"],
    number_columns=["position", "number_of_laps"],
)
stints_raw = load_raw("stints", columns=["session_key", "driver_number", "stint_number", "lap_start",
                                         "lap_end", "compound", "tyre_age_at_start"])
pit_raw = load_raw(
    "pit",
    columns=["session_key", "driver_number", "lap_number", "date", "pit_duration", "lane_duration",
             "stop_duration"],
    number_columns=["pit_duration", "lane_duration", "stop_duration"],
    date_columns=["date"],
)
race_control = load_raw("race_control", columns=["session_key", "date", "category", "message"],
                        date_columns=["date"])

print(f"races: {len(races)}")
for name, table in [("laps", laps_raw), ("drivers", drivers), ("results", results),
                    ("stints", stints_raw), ("pit", pit_raw), ("race_control", race_control)]:
    print(f"{name}: {len(table):,} rows, {table['session_key'].nunique()} races")

races: 91
laps: 101,988 rows, 91 races
drivers: 1,859 rows, 91 races
results: 1,859 rows, 91 races
stints: 5,301 rows, 91 races
pit: 3,202 rows, 85 races
race_control: 10,164 rows, 91 races


## 2. One row per driver per lap

Year and circuit come from the race index, the driver code and team from the drivers table **of the same race** (four drivers changed team during a season). Circuits are identified by the circuit name, never by the country.

In [3]:
laps = laps_raw.merge(races[["session_key", "year", "circuit"]], on="session_key", how="left",
                      validate="many_to_one")
driver_names = drivers.rename(columns={"name_acronym": "driver", "team_name": "team"})
laps = laps.merge(driver_names, on=KEY, how="left", validate="many_to_one")
laps = laps.sort_values(LAP_KEY).reset_index(drop=True)

rows_raw = len(laps)
exact_duplicates = int(laps.duplicated().sum())
laps = laps.drop_duplicates().reset_index(drop=True)
add_step("combine the raw lap files of all races", rows_raw, 0, "none", "starting point")
add_step("exact duplicate rows", rows_raw, exact_duplicates, "removed", "the same row delivered twice")

assert not laps.duplicated(LAP_KEY).any(), "a driver has two rows for the same lap"
assert laps["driver"].notna().all() and laps["team"].notna().all(), "a lap has no driver code or team"
assert drivers.groupby(KEY)["team_name"].nunique().max() == 1, "a driver has two teams in one race"

print(f"lap rows: {len(laps):,}, exact duplicates removed: {exact_duplicates}")

lap rows: 101,988, exact duplicates removed: 0


Three races have 19 cars instead of 20. The missing car is the team-mate of the only driver whose team has one car in that race.

In [4]:
cars = drivers.groupby("session_key")["driver_number"].nunique()
normal_cars = drivers.merge(races[["session_key", "year"]], on="session_key").groupby("year")["session_key"].apply(
    lambda keys: cars[keys.unique()].max())
team_size = drivers.groupby(["session_key", "team_name"]).size()
one_car_teams = team_size[team_size == 1].reset_index()[["session_key", "team_name"]]
for row in one_car_teams.merge(races, on="session_key").sort_values("date").itertuples():
    # The missing car: a driver who drove for this team in another race of the season but not here.
    same_season = races.loc[races["year"] == row.year, "session_key"]
    team_drivers = drivers[drivers["session_key"].isin(same_season) & (drivers["team_name"] == row.team_name)]
    here = set(drivers.loc[drivers["session_key"] == row.session_key, "driver_number"])
    absent = team_drivers[~team_drivers["driver_number"].isin(here)]
    absent = absent.groupby(["driver_number", "name_acronym"]).size().sort_values(ascending=False)
    number, code = absent.index[0]
    print(f"{row.year} {row.circuit}: {cars[row.session_key]} cars, {row.team_name} has one car; "
          f"missing car {number} ({code})")

2023 Singapore: 19 cars, Aston Martin has one car; missing car 18 (STR)
2024 Melbourne: 19 cars, Williams has one car; missing car 2 (SAR)
2025 Catalunya: 19 cars, Aston Martin has one car; missing car 18 (STR)


## 3. Race distance, and rows after a driver's last lap

The race distance is the winner's number of laps in the results table. `classified_laps` is the driver's own number of completed laps there.

The lap file has rows beyond `classified_laps`:
- empty rows after the chequered flag for cars that finished: **removed** (no lap was driven);
- an empty lap row for a car that did not start: **removed**;
- the lap on which a car retired: **kept**, flagged `incomplete_lap`;
- any timed lap beyond `classified_laps` of a car that finished: **kept**, flagged `beyond_classified` (a lap after the chequered flag, never a racing lap).

In [5]:
winner_laps = results[results["position"] == 1].set_index("session_key")["number_of_laps"]
assert winner_laps.index.is_unique and len(winner_laps) == len(races), "not exactly one winner per race"

result_columns = results[KEY + ["number_of_laps", "dnf", "dns", "dsq"]].rename(
    columns={"number_of_laps": "classified_laps"})
laps = laps.merge(result_columns, on=KEY, how="left", validate="many_to_one")
assert laps["dnf"].notna().all(), "a driver with laps is not in the results"
laps["race_laps"] = laps["session_key"].map(winner_laps).astype(int)

time_columns = ["lap_duration", "duration_sector_1", "duration_sector_2", "duration_sector_3"]
no_times = laps[time_columns].isna().all(axis=1)
after_last_lap = laps["lap_number"] > laps["classified_laps"]

empty_after_flag = after_last_lap & no_times & ~laps["dnf"] & ~laps["dns"]
empty_non_starter = no_times & laps["dns"]
print("Rows after the driver's last classified lap:", int(after_last_lap.sum()))
print("  empty rows after the chequered flag (removed):", int(empty_after_flag.sum()))
print("  empty rows of cars that did not start (removed):", int(empty_non_starter.sum()))
print(laps.loc[empty_non_starter, ["year", "circuit", "driver", "lap_number", "classified_laps"]].to_string(index=False))

rows_before = len(laps)
add_step("empty rows after the chequered flag", rows_before, empty_after_flag.sum(), "removed",
         "row beyond the driver's classified laps with no lap or sector time; the car had finished")
add_step("empty rows of cars that did not start", rows_before, empty_non_starter.sum(), "removed",
         "lap row with no times for a driver classified as did not start")
laps = laps[~(empty_after_flag | empty_non_starter)].reset_index(drop=True)

after_last_lap = laps["lap_number"] > laps["classified_laps"]
laps["incomplete_lap"] = after_last_lap & laps["dnf"]
laps["beyond_classified"] = after_last_lap & ~laps["dnf"]
laps["disqualified"] = laps["dsq"]
laps["no_lap_time"] = laps["lap_duration"].isna()
add_step("retirement laps", len(laps), laps["incomplete_lap"].sum(), "flagged incomplete_lap",
         "the lap on which a car retired; it was started but not completed")
add_step("timed laps beyond the classified laps", len(laps), laps["beyond_classified"].sum(),
         "flagged beyond_classified", "lap after the chequered flag; never a racing lap")
add_step("laps of disqualified drivers", len(laps), laps["disqualified"].sum(), "flagged disqualified",
         "driver disqualified; the results give no number of laps")
add_step("laps without a lap time", len(laps), laps["no_lap_time"].sum(), "flagged no_lap_time",
         "lap_duration missing at the source (mostly lap 1 and retirement laps)")

print("\nTimed laps beyond the classified laps:")
print(laps.loc[laps["beyond_classified"], ["year", "circuit", "driver", "lap_number", "lap_duration",
                                           "classified_laps", "race_laps"]].to_string(index=False))

Rows after the driver's last classified lap: 300
  empty rows after the chequered flag (removed): 43
  empty rows of cars that did not start (removed): 15
 year     circuit driver  lap_number  classified_laps
 2023       Monza    TSU           1              0.0
 2023  Interlagos    LEC           1              0.0
 2024 Silverstone    GAS           1              0.0
 2024  Interlagos    STR           1              0.0
 2025   Melbourne    HAD           1              0.0
 2025       Monza    HUL           1              0.0
 2025 Silverstone    COL           1              0.0
 2025   Spielberg    SAI           1              0.0
 2026   Melbourne    HUL           1              0.0
 2026   Melbourne    PIA           1              0.0
 2026    Shanghai    BOR           1              0.0
 2026    Shanghai    PIA           1              0.0
 2026    Montreal    LIN           1              0.0
 2026  Interlagos    BOT           1              0.0
 2026   Las Vegas    ALB           

### Laps missing at the source

For some cars the lap file has fewer rows than the results say they completed. OpenF1 simply does not have these laps (checked by asking the API again). `race_missing_laps` is the number of such laps in the race; it is 0 for a complete race. Later stages decide whether to use the affected races.

In [6]:
per_car = laps.groupby(KEY).agg(classified_laps=("classified_laps", "first"), last_lap=("lap_number", "max"))
# Disqualified drivers have no classified laps; their own last lap is used instead.
per_car["expected"] = per_car["classified_laps"].fillna(per_car["last_lap"])
completed = laps[~laps["incomplete_lap"] & ~laps["beyond_classified"]].groupby(KEY).size()
per_car["rows"] = completed.reindex(per_car.index).fillna(0)
per_car["missing"] = (per_car["expected"] - per_car["rows"]).astype(int)
assert (per_car["missing"] >= 0).all(), "a car has more completed laps than the results say"

race_missing = per_car.groupby(level="session_key")["missing"].sum()
laps["race_missing_laps"] = laps["session_key"].map(race_missing).astype(int)
incomplete_races = races[races["session_key"].isin(race_missing[race_missing > 0].index)].copy()
incomplete_races["missing_laps"] = incomplete_races["session_key"].map(race_missing)
incomplete_races["cars_affected"] = incomplete_races["session_key"].map(
    per_car[per_car["missing"] > 0].groupby(level="session_key").size())
print(incomplete_races[["year", "circuit", "session_key", "missing_laps", "cars_affected"]].to_string(index=False))
add_step("laps in races with laps missing at the source", len(laps), (laps["race_missing_laps"] > 0).sum(),
         "flagged race_missing_laps > 0",
         f"OpenF1 has no rows for {int(race_missing.sum())} completed laps in these races")

 year circuit  session_key  missing_laps  cars_affected
 2023   Monza         9157             7              7
 2025   Miami        10033           446             19


## 4. Lap end time

The lap end is the next lap's `date_start`. When there is no next lap, or the next lap's start is not later than this lap's start (lap 1 of races restarted after an early red flag carries the restart time), it is `date_start + lap_duration`.

Missing lap-1 start times stay empty (flag `date_start_missing`); the end of those laps still comes from lap 2. A lap is flagged `time_mismatch` when the next lap's start differs from `date_start + lap_duration` by more than 0.5 s.

In [7]:
by_car = laps.groupby(KEY)
next_lap_number = by_car["lap_number"].shift(-1)
next_start = by_car["date_start"].shift(-1)
# Only the lap that really follows counts (2025 Miami jumps from lap 1 to lap 25).
next_start = next_start.where(next_lap_number == laps["lap_number"] + 1)

own_end = laps["date_start"] + pd.to_timedelta(laps["lap_duration"], unit="s")
goes_backwards = next_start <= laps["date_start"]
laps["lap_end"] = next_start.where(~goes_backwards).fillna(own_end)

difference = (next_start - own_end).dt.total_seconds()
laps["time_mismatch"] = (difference.abs() > TIME_MISMATCH_SECONDS) | goes_backwards
laps["date_start_missing"] = laps["date_start"].isna()

print("laps without date_start:", int(laps["date_start_missing"].sum()),
      "| all on lap 1:", bool((laps.loc[laps["date_start_missing"], "lap_number"] == 1).all()))
print("laps whose next lap starts earlier:", int(goes_backwards.sum()))
print(f"laps flagged time_mismatch (more than {TIME_MISMATCH_SECONDS} s):", int(laps["time_mismatch"].sum()))
print("laps without a lap end time:", int(laps["lap_end"].isna().sum()))
add_step("laps without a start time", len(laps), laps["date_start_missing"].sum(), "flagged date_start_missing",
         "date_start missing at the source (lap 1 only); left empty")
add_step("lap end time disagrees with the lap time", len(laps), laps["time_mismatch"].sum(), "flagged time_mismatch",
         f"next lap's start differs from date_start + lap_duration by more than {TIME_MISMATCH_SECONDS} s, "
         "or is not later than date_start")

laps without date_start: 52 | all on lap 1: True
laps whose next lap starts earlier: 21
laps flagged time_mismatch (more than 0.5 s): 261
laps without a lap end time: 261


## 5. Red-flag periods

A red-flag period runs from a `SESSION ABORTED` message to the next `SESSION STARTED` message in race control. Laps that overlap such a period are flagged `red_flag_lap`: their lap time includes the stoppage.

In [8]:
status = race_control[race_control["category"] == "SessionStatus"].sort_values(["session_key", "date"])
periods = []
for session_key, messages in status.groupby("session_key"):
    aborted_at = None
    for message in messages.itertuples():
        if message.message == "SESSION ABORTED":
            aborted_at = message.date
        elif message.message == "SESSION STARTED" and aborted_at is not None:
            periods.append({"session_key": session_key, "red_start": aborted_at, "red_end": message.date})
            aborted_at = None
red_flags = pd.DataFrame(periods).merge(races[["session_key", "year", "circuit"]], on="session_key")
print(red_flags[["year", "circuit", "red_start", "red_end"]].to_string(index=False))


def overlaps_red_flag(table, start_column, end_column):
    """Return True for rows whose time window overlaps a red-flag period of the same race."""
    result = pd.Series(False, index=table.index)
    for period in red_flags.itertuples():
        result |= ((table["session_key"] == period.session_key)
                   & (table[start_column] <= period.red_end)
                   & (table[end_column] >= period.red_start))
    return result


laps["red_flag_lap"] = overlaps_red_flag(laps, "date_start", "lap_end")
print("\nlaps overlapping a red-flag period:", int(laps["red_flag_lap"].sum()))
print("laps longer than 300 s:", int((laps["lap_duration"] > 300).sum()),
      "| of which flagged red_flag_lap:", int(((laps["lap_duration"] > 300) & laps["red_flag_lap"]).sum()))
add_step("laps during a red flag", len(laps), laps["red_flag_lap"].sum(), "flagged red_flag_lap",
         "the lap overlaps a red-flag period, so its time includes the stoppage")

 year     circuit                        red_start                          red_end
 2023   Melbourne 2023-04-02 05:17:02.518000+00:00 2023-04-02 05:33:00.429000+00:00
 2023   Melbourne 2023-04-02 06:41:07.569000+00:00 2023-04-02 06:56:00.419000+00:00
 2023   Melbourne 2023-04-02 07:00:26.572000+00:00 2023-04-02 07:33:00.456000+00:00
 2023   Zandvoort 2023-08-27 14:31:15.895000+00:00 2023-08-27 15:14:00.131000+00:00
 2023 Mexico City 2023-10-29 20:51:52.582000+00:00 2023-10-29 21:13:00.310000+00:00
 2023  Interlagos 2023-11-05 17:06:35.195000+00:00 2023-11-05 17:31:00.366000+00:00
 2024      Suzuka 2024-04-07 05:04:24.473000+00:00 2024-04-07 05:32:00.407000+00:00
 2024 Monte Carlo 2024-05-26 13:04:07.561000+00:00 2024-05-26 13:44:00.121000+00:00
 2024  Interlagos 2024-11-03 16:37:21.752000+00:00 2024-11-03 17:02:00.390000+00:00
 2026 Monte Carlo 2026-06-07 14:35:06.971000+00:00 2026-06-07 15:12:00.071000+00:00
 2026   Zandvoort 2026-08-23 13:05:27.956000+00:00 2026-08-23 13:33:00.08800

## 6. Stops: in-laps and out-laps

**Pit records (85 races):** a pit record is a stop. Its `lap_number` is the in-lap and the next lap is the out-lap (the pit record's `date` is the pit exit time).
- Records timed before the scheduled race start are removed.
- When the lap running at the record's timestamp is not the lap after its `lap_number`, and that running lap is flagged as an out-lap in the lap file, the lap number is wrong: the in-lap is then taken from the timestamp (flag `lap_from_date`).

**Out-lap flags (every race):** an out-lap flag (`is_pit_out_lap`, lap 2 or later) with no pit record on the lap before is also a stop, if it is confirmed by a stint change (checked in section 7). The in-lap is the lap before, and the stop has no durations. This gives the stops of the six 2023 races without pit data, and a few stops that the pit data of other races misses.

The `is_pit_out_lap` flag on lap 1 is ignored: it is set for the whole grid in two races.

In [9]:
print("is_pit_out_lap on lap 1 (ignored):")
print(laps[(laps["lap_number"] == 1) & laps["is_pit_out_lap"]].groupby(["year", "circuit"]).size().to_string())

pit = pit_raw.rename(columns={"lap_number": "in_lap", "date": "pit_date"})
# Lap 1 of a restarted race carries the restart time, so the scheduled start
# from the race index is used to find records from before the race.
scheduled_start = pd.to_datetime(races.set_index("session_key")["date"], format="ISO8601", utc=True)
before_race = pit["pit_date"] < pit["session_key"].map(scheduled_start)
print("\npit records:", len(pit), "| timed before the scheduled race start (removed):", int(before_race.sum()),
      "| of which with a duration:", int(pit.loc[before_race, "pit_duration"].notna().sum()))
pit = pit[~before_race].reset_index(drop=True)


def add_lap_window(stops):
    """Add the start of the in-lap and the end of the out-lap (the other lap is used when one is missing)."""
    stops = stops.drop(columns=["window_start", "window_end"], errors="ignore")
    lap_times = laps[LAP_KEY + ["date_start", "lap_end"]]
    in_lap = lap_times.rename(columns={"lap_number": "in_lap", "date_start": "in_start", "lap_end": "in_end"})
    out_lap = lap_times.rename(columns={"lap_number": "out_lap", "date_start": "out_start", "lap_end": "out_end"})
    stops = stops.assign(out_lap=stops["in_lap"] + 1)
    stops = stops.merge(in_lap, on=KEY + ["in_lap"], how="left").merge(out_lap, on=KEY + ["out_lap"], how="left")
    stops["window_start"] = stops["in_start"].fillna(stops["out_start"])
    stops["window_end"] = stops["out_end"].fillna(stops["in_end"])
    return stops.drop(columns=["in_start", "in_end", "out_start", "out_end"])


# The pit record's date is the pit exit, so it lies in the out-lap. Find the
# lap that was running at that time and compare it with the record's lap number.
running_lap = []
running_is_out_lap = []
for record in pit.itertuples():
    car_laps = laps[(laps["session_key"] == record.session_key) & (laps["driver_number"] == record.driver_number)
                    & (laps["date_start"] <= record.pit_date)]
    if len(car_laps) == 0:
        running_lap.append(np.nan)
        running_is_out_lap.append(False)
    else:
        running = car_laps.loc[car_laps["date_start"].idxmax()]
        running_lap.append(running["lap_number"])
        running_is_out_lap.append(bool(running["is_pit_out_lap"]))
pit["lap_at_pit_date"] = running_lap
contradicts = pit["lap_at_pit_date"].notna() & (pit["lap_at_pit_date"] != pit["in_lap"] + 1)
# The timestamp is trusted only when the lap file confirms it with an out-lap
# flag. Without that, the lap number is kept (laps missing at the source and
# restarted first laps make the timestamp comparison unreliable).
pit["lap_from_date"] = contradicts & pd.Series(running_is_out_lap, index=pit.index)
pit["in_lap_delivered"] = pit["in_lap"]
pit.loc[pit["lap_from_date"], "in_lap"] = pit.loc[pit["lap_from_date"], "lap_at_pit_date"].astype(int) - 1
pit = add_lap_window(pit)

with_names = pit.merge(races[["session_key", "year", "circuit"]], on="session_key")
print("pit records whose lap number contradicts the timestamp, by race:")
print(with_names[contradicts.values].groupby(["year", "circuit", "lap_from_date"]).size().to_string())
print("\nin-lap taken from the timestamp (lap_from_date):", int(pit["lap_from_date"].sum()))
print(with_names.loc[pit["lap_from_date"].values, ["year", "circuit", "driver_number", "in_lap_delivered", "in_lap",
                                                   "pit_date", "pit_duration"]].to_string(index=False))
assert not pit.duplicated(KEY + ["in_lap"]).any(), "two pit records on the same lap of one car"
pit["source"] = "pit"

is_pit_out_lap on lap 1 (ignored):
year  circuit    
2024  Montreal       20
2026  Mexico City    22

pit records: 3202 | timed before the scheduled race start (removed): 9 | of which with a duration: 0


pit records whose lap number contradicts the timestamp, by race:
year  circuit     lap_from_date
2023  Interlagos  False             2
2024  Suzuka      False            18
2025  Catalunya   False             1
2026  Melbourne   True             10

in-lap taken from the timestamp (lap_from_date): 10
 year   circuit  driver_number  in_lap_delivered  in_lap                         pit_date  pit_duration
 2026 Melbourne              1                46      11 2026-03-08 04:19:43.406000+00:00        18.266
 2026 Melbourne             10                46      11 2026-03-08 04:19:54.135000+00:00        18.672
 2026 Melbourne             31                46      11 2026-03-08 04:19:56.923000+00:00        18.570
 2026 Melbourne             55                46      11 2026-03-08 04:20:04.992000+00:00        19.859
 2026 Melbourne             30                46      11 2026-03-08 04:20:20.833000+00:00        19.112
 2026 Melbourne              5                18      33 2026-03-08 04:53:

In [10]:
# Stops from the out-lap flag, where no pit record covers them.
races_with_pit = set(pit_raw["session_key"])
races_without_pit = races[~races["session_key"].isin(races_with_pit)]
print("races without pit data:")
print(races_without_pit[["year", "circuit", "session_key"]].to_string(index=False))

flagged_out_laps = laps[laps["is_pit_out_lap"] & (laps["lap_number"] > 1)]
flag_stops = flagged_out_laps[KEY].copy()
flag_stops["in_lap"] = flagged_out_laps["lap_number"] - 1
has_pit_record = flag_stops.merge(pit[KEY + ["in_lap"]], on=KEY + ["in_lap"], how="left", indicator=True)["_merge"] == "both"
flag_stops = flag_stops[~has_pit_record.values]
print("\nout-lap flags without a pit record on the lap before:", len(flag_stops),
      "| in races with pit data:", int(flag_stops["session_key"].isin(races_with_pit).sum()))
flag_stops["source"] = "out_lap_flag"
flag_stops["lap_from_date"] = False
flag_stops = add_lap_window(flag_stops)

stops = pd.concat([pit, flag_stops], ignore_index=True)
stops["out_lap"] = stops["in_lap"] + 1

# A stop is a red-flag stop when its in-lap to out-lap window, or the time
# the car spent in the pit lane, overlaps a red-flag period.
stops["pit_entry"] = stops["pit_date"] - pd.to_timedelta(stops["pit_duration"], unit="s")
stops["red_flag"] = (overlaps_red_flag(stops, "window_start", "window_end")
                     | overlaps_red_flag(stops, "pit_entry", "pit_date"))
print("\nstops by source and red flag:")
print(stops.groupby(["source", "red_flag"]).size().to_string())

races without pit data:
 year     circuit  session_key
 2023      Sakhir         7953
 2023      Jeddah         7779
 2023   Melbourne         7787
 2023        Baku         9070
 2023       Miami         9078
 2023 Monte Carlo         9094

out-lap flags without a pit record on the lap before: 225 | in races with pit data: 6

stops by source and red flag:
source        red_flag
out_lap_flag  False        169
              True          56
pit           False       2959
              True         234


## 7. Stints: compound and tyre age

Each change of stint is matched to a stop of the same car:
1. normally the stop's in-lap is the lap **before** the new stint;
2. at Montreal, Melbourne and Monte Carlo the stop happens before the timing line, and the stints data starts the new stint **on the in-lap**. There the new stint is moved one lap later, to the out-lap, as at every other circuit (flag `stint_shifted`).

After this, stints that cover no lap (a set fitted and removed without a completed lap) are dropped from the lap assignment and counted.

A stint change is the `same_set` when the compound is unchanged and the tyre age simply carries on. This happens when the field is led through the pit lane behind the safety car, during red flags, and at stops without a tyre change.

In [11]:
stints = stints_raw.sort_values(KEY + ["stint_number"]).reset_index(drop=True)
by_car = stints.groupby(KEY)
previous_compound = by_car["compound"].shift(1)
previous_age = by_car["tyre_age_at_start"].shift(1)
previous_start = by_car["lap_start"].shift(1)
stints["is_change"] = stints["stint_number"] > 1
stints["same_set"] = ((stints["compound"] == previous_compound)
                      & (stints["tyre_age_at_start"] == previous_age + stints["lap_start"] - previous_start))
stints["lap_start_delivered"] = stints["lap_start"]
print("stints:", len(stints), "| stint changes:", int(stints["is_change"].sum()),
      "| covering no lap as delivered:", int((stints["lap_end"] < stints["lap_start"]).sum()))

# Match every stint change to a stop, going through each car's stints in order.
stop_laps = stops.groupby(KEY)["in_lap"].apply(set).to_dict()
matched_in_lap = []
for car, car_stints in stints.groupby(KEY, sort=False):
    free_stops = set(stop_laps.get(car, set()))
    for stint in car_stints.itertuples():
        match = np.nan
        if stint.is_change:
            if stint.lap_start - 1 in free_stops:
                match = stint.lap_start - 1
            elif stint.lap_start in free_stops:
                match = stint.lap_start
            if not np.isnan(match):
                free_stops.remove(match)
        matched_in_lap.append(match)
stints["stop_in_lap"] = matched_in_lap
stints["stint_shifted"] = stints["stop_in_lap"] == stints["lap_start"]

# Move the shifted stints to the out-lap; each stint then ends where the next one starts.
stints["lap_start"] = stints["lap_start"] + stints["stint_shifted"].astype(int)
next_stint_start = stints.groupby(KEY)["lap_start"].shift(-1)
stints["lap_end"] = (next_stint_start - 1).fillna(stints["lap_end"]).astype(int)

shifted = stints[stints["stint_shifted"]].merge(races[["session_key", "year", "circuit"]], on="session_key")
print("\nstints moved one lap later (stint_shifted):", len(shifted))
print(shifted.groupby("circuit").size().sort_values(ascending=False).to_string())

no_lap = stints["lap_end"] < stints["lap_start"]
print("\nstints covering no lap after the move (not assigned to any lap):", int(no_lap.sum()))
stints = stints[~no_lap].reset_index(drop=True)
stints["is_first"] = stints.groupby(KEY).cumcount() == 0

stints: 5301 | stint changes: 3445 | covering no lap as delivered: 43



stints moved one lap later (stint_shifted): 584
circuit
Melbourne       205
Montreal        189
Monte Carlo     185
Kuala Lumpur      5

stints covering no lap after the move (not assigned to any lap): 11


### Stint changes without a stop

A stint change with no stop on the lap before or on the same lap is listed as an exception:
- `red_flag`: the change falls in a red-flag period (tyres changed in the pit lane during the stoppage, no pit record);
- `no_stop_found`: the stints data puts the boundary on a lap where the car did not stop. The data is kept as delivered and the laps between this boundary and the car's nearest stop without a stint change are flagged `compound_uncertain` (the whole new stint when the car has no such stop). The stop gets the same flag in the pit-stop table.

In [12]:
exceptions = stints[~stints["is_first"] & stints["stop_in_lap"].isna()].copy()
exceptions["in_lap"] = exceptions["lap_start"] - 1
exceptions = add_lap_window(exceptions)
exceptions["reason"] = np.where(overlaps_red_flag(exceptions, "window_start", "window_end"),
                                "red_flag", "no_stop_found")

# Stops that were not matched to any stint change.
matched_stops = stints.loc[stints["stop_in_lap"].notna(), KEY + ["stop_in_lap", "same_set", "stint_shifted"]]
matched_stops = matched_stops.rename(columns={"stop_in_lap": "in_lap"})
stops = stops.merge(matched_stops, on=KEY + ["in_lap"], how="left", validate="one_to_one")
stops["has_stint_change"] = stops["same_set"].notna()
stops["stint_shifted"] = stops["stint_shifted"].eq(True)
stops["compound_uncertain"] = ~stops["has_stint_change"] & ~stops["red_flag"]

# A stop taken from an out-lap flag must be confirmed by a stint change.
unconfirmed = (stops["source"] == "out_lap_flag") & ~stops["has_stint_change"]
print("out-lap flags without a pit record and without a stint change (not counted as stops):",
      int(unconfirmed.sum()))
stops = stops[~unconfirmed].reset_index(drop=True)

# Laps between a misplaced boundary and the nearest unmatched stop of the same car.
first_flagged, last_flagged, paired_stop = [], [], []
for change in exceptions.itertuples():
    first_lap, last_lap, stop_lap = np.nan, np.nan, np.nan
    if change.reason == "no_stop_found":
        first_lap, last_lap = change.lap_start, change.lap_end
        candidates = stops[(stops["session_key"] == change.session_key)
                           & (stops["driver_number"] == change.driver_number)
                           & stops["compound_uncertain"]]
        if len(candidates) > 0:
            nearest = (candidates["out_lap"] - change.lap_start).abs().idxmin()
            stop_lap = candidates.loc[nearest, "in_lap"]
            first_lap = min(change.lap_start, stop_lap + 1)
            last_lap = max(change.lap_start, stop_lap + 1) - 1
    first_flagged.append(first_lap)
    last_flagged.append(last_lap)
    paired_stop.append(stop_lap)
exceptions["nearest_unmatched_stop_in_lap"] = paired_stop
exceptions["first_uncertain_lap"] = first_flagged
exceptions["last_uncertain_lap"] = last_flagged

exceptions = exceptions.merge(races[["session_key", "year", "circuit"]], on="session_key")
exceptions = exceptions.merge(driver_names, on=KEY, how="left")
exception_columns = ["year", "circuit", "session_key", "driver_number", "driver", "stint_number", "lap_start",
                     "lap_end", "compound", "tyre_age_at_start", "same_set", "reason",
                     "nearest_unmatched_stop_in_lap", "first_uncertain_lap", "last_uncertain_lap"]
exceptions = exceptions[exception_columns].sort_values(["year", "session_key", "driver_number", "stint_number"])
exceptions = exceptions.reset_index(drop=True)
print("stint changes without a matching stop:", len(exceptions))
print(exceptions.groupby("reason").size().to_string())
print("\nby race (changes, of which paired with a stop that has no stint change):")
print(exceptions.groupby(["year", "circuit", "reason"]).agg(
    changes=("lap_start", "size"), paired=("nearest_unmatched_stop_in_lap", "count")).to_string())
print("\nstops without a stint change:", int((~stops["has_stint_change"]).sum()),
      "| of which during a red flag:", int((~stops["has_stint_change"] & stops["red_flag"]).sum()),
      "| flagged compound_uncertain:", int(stops["compound_uncertain"].sum()))

out-lap flags without a pit record and without a stint change (not counted as stops): 0


stint changes without a matching stop: 87
reason
no_stop_found    87

by race (changes, of which paired with a stop that has no stint change):
                                      changes  paired
year circuit           reason                        
2023 Montreal          no_stop_found        1       1
2024 Lusail            no_stop_found        2       0
     Spa-Francorchamps no_stop_found        4       0
2025 Las Vegas         no_stop_found        1       0
     Melbourne         no_stop_found        6       6
     Miami             no_stop_found       15      15
     Spa-Francorchamps no_stop_found       20       6
2026 Austin            no_stop_found        1       1
     Catalunya         no_stop_found        1       1
     Hungaroring       no_stop_found        1       1
     Interlagos        no_stop_found        1       1
     Kuala Lumpur      no_stop_found       27      26
     Las Vegas         no_stop_found        1       1
     Mexico City       no_stop_found        2  

### One row per stint lap, joined to the laps

`tyre_age` is the number of laps already done on the set when the lap starts, including laps from earlier sessions (`tyre_age_at_start`). `used_set` says that the set was not new when it was fitted.

Laps without tyre information keep an empty compound and tyre age, and `tyre_flag` says why: `no_stint` (no stint covers the lap) or `unknown_compound` (the stint's compound is `UNKNOWN`). `tyre_flag` is `compound_uncertain` for the laps described above; they keep the delivered compound.

In [13]:
stints["laps_in_stint"] = stints["lap_end"] - stints["lap_start"] + 1
stint_laps = stints.loc[stints.index.repeat(stints["laps_in_stint"])].copy()
laps_into_stint = stint_laps.groupby(level=0).cumcount()
stint_laps["lap_number"] = stint_laps["lap_start"] + laps_into_stint
stint_laps["tyre_age"] = stint_laps["tyre_age_at_start"] + laps_into_stint
stint_laps["used_set"] = stint_laps["tyre_age_at_start"] > 0

# One car has two stints claiming the same lap; the later stint is kept
# because it is the later message of the tyre data.
overlapping = stint_laps.duplicated(LAP_KEY, keep="last")
print("stint laps claimed by two stints (the later stint is kept):", int(overlapping.sum()))
print(stint_laps.loc[stint_laps.duplicated(LAP_KEY, keep=False),
                     LAP_KEY + ["stint_number", "compound", "tyre_age"]].to_string(index=False))
stint_laps = stint_laps[~overlapping]
stint_laps = stint_laps[LAP_KEY + ["stint_number", "compound", "tyre_age", "used_set", "stint_shifted"]]

laps = laps.merge(stint_laps, on=LAP_KEY, how="left", validate="one_to_one")

laps["tyre_flag"] = ""
laps.loc[laps["stint_number"].isna(), "tyre_flag"] = "no_stint"
unknown_compound = laps["compound"] == "UNKNOWN"
laps.loc[unknown_compound, "tyre_flag"] = "unknown_compound"
laps.loc[unknown_compound, "compound"] = None
for change in exceptions[exceptions["reason"] == "no_stop_found"].itertuples():
    in_range = ((laps["session_key"] == change.session_key) & (laps["driver_number"] == change.driver_number)
                & laps["lap_number"].between(change.first_uncertain_lap, change.last_uncertain_lap)
                & (laps["tyre_flag"] == ""))
    laps.loc[in_range, "tyre_flag"] = "compound_uncertain"

laps["stint_number"] = laps["stint_number"].astype("Int64")
laps["tyre_age"] = laps["tyre_age"].astype("Int64")
laps["used_set"] = laps["used_set"].astype("boolean")
laps["stint_shifted"] = laps["stint_shifted"].eq(True)

print("\ncompound:")
print(laps["compound"].value_counts(dropna=False).to_string())
print("\ntyre_flag:")
print(laps["tyre_flag"].replace("", "(none)").value_counts().to_string())
add_step("stints covering no lap", len(laps), 0, "none",
         f"{int(no_lap.sum())} stints (a set fitted and removed without a completed lap) are assigned to no lap")
add_step("stint moved to the out-lap", len(laps), laps["stint_shifted"].sum(), "flagged stint_shifted",
         f"{len(shifted)} stints started on the in-lap in the stints data (stop before the timing line)")
add_step("laps without a stint", len(laps), (laps["tyre_flag"] == "no_stint").sum(), "flagged tyre_flag = no_stint",
         "no stint covers the lap (almost all are retirement laps); compound and tyre age empty")
add_step("laps with an unknown compound", len(laps), (laps["tyre_flag"] == "unknown_compound").sum(),
         "flagged tyre_flag = unknown_compound", "the stints data says UNKNOWN; compound set to empty")
add_step("laps with an uncertain compound", len(laps), (laps["tyre_flag"] == "compound_uncertain").sum(),
         "flagged tyre_flag = compound_uncertain",
         f"{int((exceptions['reason'] == 'no_stop_found').sum())} stint changes on a lap without a stop; "
         "compound and tyre age kept as delivered")

stint laps claimed by two stints (the later stint is kept): 6
 session_key  driver_number  lap_number  stint_number     compound  tyre_age
        9693              5           3             2 INTERMEDIATE         2
        9693              5           3             4 INTERMEDIATE         3
        9693              5           4             4 INTERMEDIATE         4
        9693              5           4             6       MEDIUM         0
        9693             30           2             1 INTERMEDIATE         1
        9693             30           3             2 INTERMEDIATE         0
        9693             30           2             4 INTERMEDIATE         2
        9693             30           3             5 INTERMEDIATE         3
        9693             87           3             2 INTERMEDIATE         2
        9693             87           3             4 INTERMEDIATE         0
        9693             87           4             4 INTERMEDIATE         1
        9693  


compound:
compound
HARD            45245
MEDIUM          35737
SOFT            15898
INTERMEDIATE     4753
NaN               209
WET                84
None                4

tyre_flag:
tyre_flag
(none)                100587
compound_uncertain      1130
no_stint                 209
unknown_compound           4


## 8. Mark in-laps and out-laps

Every stop marks its in-lap and its out-lap, including stops during a red flag (those laps also carry `red_flag_lap`).

In [14]:
in_laps = stops[KEY + ["in_lap"]].rename(columns={"in_lap": "lap_number"}).assign(is_in_lap=True)
out_laps = stops[KEY + ["out_lap"]].rename(columns={"out_lap": "lap_number"}).assign(is_out_lap=True)
laps = laps.merge(in_laps, on=LAP_KEY, how="left", validate="one_to_one")
laps = laps.merge(out_laps, on=LAP_KEY, how="left", validate="one_to_one")
laps["is_in_lap"] = laps["is_in_lap"].eq(True)
laps["is_out_lap"] = laps["is_out_lap"].eq(True)
add_step("in-laps", len(laps), laps["is_in_lap"].sum(), "flagged is_in_lap", "the lap that ends with a stop")
add_step("out-laps", len(laps), laps["is_out_lap"].sum(), "flagged is_out_lap", "the lap that follows a stop")

# Compare with the out-lap flag of the lap file (lap 2 or later, races with pit data).
flagged = laps[laps["is_pit_out_lap"] & (laps["lap_number"] > 1) & laps["session_key"].isin(races_with_pit)]
print("is_pit_out_lap flags after lap 1 in races with pit data:", len(flagged),
      "| with a stop on the lap before:", int(flagged["is_out_lap"].sum()))
print(flagged[~flagged["is_out_lap"]].groupby(["year", "circuit"]).size().to_string())
melbourne_2026 = flagged[(flagged["year"] == 2026) & (flagged["circuit"] == "Melbourne")]
print(f"2026 Melbourne: {len(melbourne_2026)} out-lap flags, {int(melbourne_2026['is_out_lap'].sum())} with a stop")
ours_only = laps[laps["is_out_lap"] & ~laps["is_pit_out_lap"]]
print("out-laps from stops that the lap file does not flag:", len(ours_only),
      "| of which the lap row exists but belongs to a red-flag stop:", int(ours_only["red_flag_lap"].sum()))

is_pit_out_lap flags after lap 1 in races with pit data: 3195 | with a stop on the lap before: 3195
Series([], )
2026 Melbourne: 32 out-lap flags, 32 with a stop
out-laps from stops that the lap file does not flag: 0 | of which the lap row exists but belongs to a red-flag stop: 0


## 9. Pit-stop table

One row per stop. `tyre_age_at_stop` is the number of laps on the old set when it comes off (its tyre age at the start of the in-lap plus one). The duration fields are empty for stops taken from the out-lap flag (the six races without pit data and a few others), and `stop_duration` only exists from 2024 on.

In [15]:
tyres_by_lap = stint_laps[LAP_KEY + ["compound", "tyre_age"]]
before = tyres_by_lap.rename(columns={"lap_number": "in_lap", "compound": "compound_before", "tyre_age": "age_in_lap"})
after = tyres_by_lap.rename(columns={"lap_number": "out_lap", "compound": "compound_after",
                                     "tyre_age": "tyre_age_after"})
pit_stops = stops.merge(before, on=KEY + ["in_lap"], how="left").merge(after, on=KEY + ["out_lap"], how="left")
pit_stops["tyre_age_at_stop"] = (pit_stops["age_in_lap"] + 1).astype("Int64")
pit_stops["tyre_age_after"] = pit_stops["tyre_age_after"].astype("Int64")
pit_stops["same_set"] = pit_stops["same_set"].astype("boolean")
pit_stops = pit_stops.merge(races[["session_key", "year", "circuit"]], on="session_key", how="left")
pit_stops = pit_stops.merge(driver_names, on=KEY, how="left", validate="many_to_one")
pit_stops = pit_stops[["year", "session_key", "circuit", "driver_number", "driver", "team", "in_lap", "out_lap",
                       "source", "pit_date", "pit_duration", "lane_duration", "stop_duration",
                       "compound_before", "compound_after", "tyre_age_at_stop", "tyre_age_after",
                       "has_stint_change", "same_set", "red_flag", "stint_shifted", "lap_from_date",
                       "compound_uncertain"]]
pit_stops = pit_stops.sort_values(["year", "session_key", "driver_number", "in_lap"]).reset_index(drop=True)

print("stops:", len(pit_stops))
print(pit_stops.groupby(["year", "source"]).size().unstack(fill_value=0).to_string())
print("\nflags (number of stops):")
print(pit_stops[["has_stint_change", "red_flag", "stint_shifted", "lap_from_date", "compound_uncertain"]].sum().to_string())
print("same_set:", int(pit_stops["same_set"].sum()))
print("\nmissing values:")
print(pit_stops.isna().sum()[lambda counts: counts > 0].to_string())

stops: 3418
source  out_lap_flag  pit
year                     
2023             219  693
2024               0  825
2025               0  821
2026               6  854

flags (number of stops):
has_stint_change      3347
red_flag               290
stint_shifted          574
lap_from_date           10
compound_uncertain      71
same_set: 266

missing values:
pit_date          225
pit_duration      225
lane_duration     225
stop_duration    1953
same_set           71


Same-set stint changes outside red flags, by race. Where the whole field has them, race control shows the safety car leading the cars through the pit lane: the cars pass the pit lane without stopping, so the set and its tyre age carry on.

In [16]:
same_set_stops = pit_stops[pit_stops["same_set"].fillna(False) & ~pit_stops["red_flag"]]
summary = same_set_stops.groupby(["year", "circuit"]).agg(
    stops=("in_lap", "size"), cars=("driver_number", "nunique"),
    first_in_lap=("in_lap", "min"), last_in_lap=("in_lap", "max"),
    median_pit_duration=("pit_duration", "median"), with_stop_duration=("stop_duration", "count"))
print(summary.to_string())
through_pit_lane = race_control[race_control["message"].str.contains("THROUGH THE PIT LANE", na=False)]
through_pit_lane = through_pit_lane.merge(races[["session_key", "year", "circuit"]], on="session_key")
print("\nrace control messages about the pit lane:")
print(through_pit_lane[["year", "circuit", "date", "message"]].to_string(index=False))

                         stops  cars  first_in_lap  last_in_lap  median_pit_duration  with_stop_duration
year circuit                                                                                            
2023 Sakhir                  1     1            15           15                  NaN                   0
     Silverstone             1     1            36           36               34.300                   0
     Spielberg              19    19             2            2               16.700                   0
     Suzuka                  1     1             2            2               32.700                   0
2024 Imola                   1     1            23           23               38.400                   0
     Las Vegas               1     1            11           11               15.700                   0
     Lusail                 34    17            35           47               26.150                   1
     Miami                   1     1            31     

## 10. Checks

In [17]:
# 1. Exactly one row per driver per lap.
assert not laps.duplicated(LAP_KEY).any()
print("1. one row per driver per lap:", len(laps), "rows,", laps[KEY].drop_duplicates().shape[0], "cars")

# 2. Every lap has a compound and tyre age, or a flag saying why not.
no_tyre = laps["compound"].isna() | laps["tyre_age"].isna()
assert (laps.loc[no_tyre, "tyre_flag"].isin(["no_stint", "unknown_compound"])).all()
assert not (laps.loc[~no_tyre, "tyre_flag"].isin(["no_stint", "unknown_compound"])).any()
print("2. laps without compound or tyre age:", int(no_tyre.sum()), "| all with a tyre_flag: True")
print(laps.loc[no_tyre].groupby(["tyre_flag", "incomplete_lap"]).size().to_string())

# 3. No driver has more laps than the winner; drivers with fewer laps are explained.
last_laps = laps.groupby(KEY).agg(year=("year", "first"), circuit=("circuit", "first"), driver=("driver", "first"),
                                  last_lap=("lap_number", "max"), race_laps=("race_laps", "first"),
                                  classified_laps=("classified_laps", "first"), dnf=("dnf", "first"),
                                  dsq=("dsq", "first"), race_missing_laps=("race_missing_laps", "first"))
assert (last_laps["last_lap"] <= last_laps["race_laps"]).all()
fewer = last_laps[last_laps["last_lap"] < last_laps["race_laps"]].copy()
fewer["why"] = np.select(
    [fewer["dnf"], fewer["dsq"], fewer["classified_laps"] < fewer["race_laps"], fewer["race_missing_laps"] > 0],
    ["retired", "disqualified", "lapped", "laps missing at the source"], default="unexplained")
print("\n3. cars with more laps than the winner: 0")
print("   cars with fewer lap rows than the winner:", len(fewer))
print(fewer["why"].value_counts().to_string())
print(fewer.loc[fewer["why"].isin(["laps missing at the source", "unexplained"]),
                ["year", "circuit", "driver", "last_lap", "classified_laps", "race_laps"]].to_string())
assert (fewer["why"] != "unexplained").all()

# 4. Every stint change has a matching stop, or is listed as an exception.
changes = stints[~stints["is_first"]]
print("\n4. stint changes (stints that cover a lap):", len(changes),
      "| with a stop:", int(changes["stop_in_lap"].notna().sum()),
      "| listed as exceptions:", len(exceptions))
assert changes["stop_in_lap"].notna().sum() + len(exceptions) == len(changes)

1. one row per driver per lap: 101930 rows, 1839 cars


2. laps without compound or tyre age: 213 | all with a tyre_flag: True
tyre_flag         incomplete_lap
no_stint          False               1
                  True              208
unknown_compound  False               2
                  True                2

3. cars with more laps than the winner: 0
   cars with fewer lap rows than the winner: 670
why
lapped                        425
retired                       237
laps missing at the source      7
disqualified                    1
                           year circuit driver  last_lap  classified_laps  race_laps
session_key driver_number                                                           
9157        2              2023   Monza    SAR        50             51.0         51
            10             2023   Monza    GAS        50             51.0         51
            18             2023   Monza    STR        50             51.0         51
            24             2023   Monza    ZHO        50             51.0      


4. stint changes (stints that cover a lap): 3434 | with a stop: 3347 | listed as exceptions: 87


## 11. Save

In [18]:
lap_columns = ["year", "session_key", "circuit", "driver_number", "driver", "team", "lap_number", "lap_duration",
               "duration_sector_1", "duration_sector_2", "duration_sector_3", "i1_speed", "i2_speed", "st_speed",
               "date_start", "lap_end", "is_pit_out_lap", "race_laps", "compound", "stint_number", "tyre_age",
               "used_set", "is_in_lap", "is_out_lap", "tyre_flag", "stint_shifted", "incomplete_lap",
               "beyond_classified", "disqualified", "no_lap_time", "date_start_missing", "time_mismatch",
               "red_flag_lap", "race_missing_laps"]
laps = laps[lap_columns].sort_values(["year", "session_key", "driver_number", "lap_number"]).reset_index(drop=True)

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
laps.to_parquet(DATA_PROCESSED / "laps.parquet", index=False)
pit_stops.to_parquet(DATA_PROCESSED / "pit_stops.parquet", index=False)
cleaning_table = pd.DataFrame(cleaning_steps)
cleaning_table.to_csv(RESULTS / "cleaning_table.csv", index=False)
exceptions.to_csv(RESULTS / "stint_change_exceptions.csv", index=False)
print("saved laps.parquet:", laps.shape, "| pit_stops.parquet:", pit_stops.shape)

saved laps.parquet: (101930, 34) | pit_stops.parquet: (3418, 23)


## 12. Report

In [19]:
pd.set_option("display.max_colwidth", 120)
print("CLEANING TABLE")
print(cleaning_table.to_string(index=False))

print("\nPROBLEM COUNTS AFTER CLEANING")
flag_columns = ["incomplete_lap", "beyond_classified", "disqualified", "no_lap_time", "date_start_missing",
                "time_mismatch", "red_flag_lap", "stint_shifted", "is_in_lap", "is_out_lap"]
print("lap rows:", len(laps), "| races:", laps["session_key"].nunique(),
      "| duplicate (race, driver, lap):", int(laps.duplicated(LAP_KEY).sum()))
print(laps[flag_columns].sum().to_string())
print("tyre_flag:", laps["tyre_flag"].replace("", "(none)").value_counts().to_dict())
print("laps in races with missing laps:", laps[laps["race_missing_laps"] > 0].groupby(
    ["year", "circuit"])["race_missing_laps"].agg(["size", "first"]).rename(
    columns={"size": "lap_rows", "first": "missing_laps"}).to_dict("index"))
print("missing values in laps:")
print(laps.isna().sum()[lambda counts: counts > 0].to_string())
print("\nlaps per season:")
print(laps.groupby("year").agg(races=("session_key", "nunique"), laps=("lap_number", "size")).to_string())

print("\nEXAMPLE ROWS: laps")
print(laps.sample(3, random_state=42).T.to_string())
print("\nEXAMPLE ROWS: pit_stops")
print(pit_stops.sample(3, random_state=42).T.to_string())

print(f"\nRunning time: {time.time() - start_time:.1f} seconds")

CLEANING TABLE
                                         step  laps_before  laps_affected                                 action                                                                                                      reason
       combine the raw lap files of all races       101988              0                                   none                                                                                              starting point
                         exact duplicate rows       101988              0                                removed                                                                                the same row delivered twice
          empty rows after the chequered flag       101988             43                                removed                    row beyond the driver's classified laps with no lap or sector time; the car had finished
        empty rows of cars that did not start       101988             15                            